In [1]:
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/reconocimiento_facial_tec_culiacan'
print('✅ Drive montado')

Mounted at /content/drive
✅ Drive montado


In [2]:
!pip install -q tf2onnx onnx onnxruntime

print('✅ Librerías instaladas')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.1/839.1 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 72.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 70.1 MB/s eta 0:00:00
✅ Librerías instaladas


In [5]:
import os

carpeta_modelo = f'{BASE}/modelo'
archivos = os.listdir(carpeta_modelo)
print('Archivos en /modelo/:')
for a in archivos:
    print(f'   {a}')


Archivos en /modelo/:
   modelo_facial.keras
   modelo_facial.tflite


In [6]:
import tf2onnx
import tensorflow as tf
import onnx

# Cargar el modelo .keras
modelo = tf.keras.models.load_model(f'{BASE}/modelo/modelo_facial.keras')
print('✅ Modelo cargado')

# Convertir a ONNX
input_signature = [tf.TensorSpec(shape=[None, 224, 224, 3], dtype=tf.float32, name='input')]
onnx_model, _ = tf2onnx.convert.from_keras(modelo, input_signature=input_signature, opset=13)
print('✅ Conversión completada')

# Guardar en Drive
ruta_onnx = f'{BASE}/modelo/modelo_facial.onnx'
onnx.save(onnx_model, ruta_onnx)
print(f'✅ Modelo guardado en: {ruta_onnx}')

✅ Modelo cargado
✅ Conversión completada
✅ Modelo guardado en: /content/drive/MyDrive/reconocimiento_facial_tec_culiacan/modelo/modelo_facial.onnx


In [7]:
import onnxruntime as ort
import numpy as np

sess = ort.InferenceSession(ruta_onnx)
input_name = sess.get_inputs()[0].name

# Prueba con imagen falsa
dummy = np.random.rand(1, 224, 224, 3).astype(np.float32)
resultado = sess.run(None, {input_name: dummy})

print(f'✅ Modelo ONNX funcionando')
print(f'   Input name: {input_name}')
print(f'   Output shape: {resultado[0].shape}')
print(f'   Clases: {resultado[0].shape[1]} (debe ser 20)')

✅ Modelo ONNX funcionando
   Input name: input
   Output shape: (1, 20)
   Clases: 20 (debe ser 20)
